# RAG Support Chatbot — Milestone 2 (VS Code / local version)
## Model Development & Evaluation — Local FAISS Vector Store

**No Azure account needed.** This version swaps Azure AI Search for:
- **Embeddings:** `sentence-transformers` (runs on your CPU, free, offline)
- **Vector store:** `FAISS` (Facebook AI Similarity Search — free, local, no server)

**Design note:** the retrieval interface (`vector_search()`, `hybrid_search()`)
is written so it's a near drop-in swap if you later reconnect Azure —
same function signatures, same return format.

**Pipeline:**
```
Step 1 → Install dependencies
Step 2 → Load train/val/test splits from Milestone 1
Step 3 → Generate embeddings locally (sentence-transformers)
Step 4 → Build the FAISS index
Step 5 → Vector search test
Step 6 → Hybrid search (vector + keyword BM25 via rank_bm25)
Step 7 → Evaluation — BLEU, ROUGE, retrieval relevance@k
Step 8 → Save the index + embeddings to disk
```

## Step 1 — Install dependencies (run once in your venv terminal)

In [2]:
# Run this once in your activated venv terminal:
#   pip install sentence-transformers faiss-cpu rank_bm25 nltk rouge-score pandas numpy tqdm
#
# Uncomment to install from inside the notebook instead:
# %pip install sentence-transformers faiss-cpu rank_bm25 nltk rouge-score pandas numpy tqdm

In [3]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import faiss
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

DATA_DIR    = "../data"
INDEX_DIR   = "../data/faiss_index"
os.makedirs(INDEX_DIR, exist_ok=True)

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"   # 384-dim, fast, strong general-purpose model
EMBED_BATCH_SIZE      = 64

print('Imports ready')
print(f'  Embedding model: {EMBEDDING_MODEL_NAME} (384 dimensions)')

c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports ready
  Embedding model: all-MiniLM-L6-v2 (384 dimensions)


## Step 2 — Load Milestone 1 splits

In [4]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_df.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val_df.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test_df.csv"))

print(f'Train : {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(f'Test  : {len(test_df):,} rows')

train_df[['category', 'intent', 'instruction_clean', 'response_clean']].head(3)

Train : 17,268 rows
Val   : 2,159 rows
Test  : 2,159 rows


,category,intent,instruction_clean,response_clean
0,ORDER,track_order,where do i check the current status of purchas...,Grateful for your contact! I get the sense tha...
1,REFUND,check_refund_policy,I don't know how I can see your refund policy,I appreciate your inquiry regarding how to acc...
2,ACCOUNT,switch_account,i need help to use the standard profile,I'll get right on it! I'm here to guide you in...


## Step 3 — Generate embeddings locally

In [5]:
print(f'Loading {EMBEDDING_MODEL_NAME}... (downloads ~90MB the first time only)')
embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print('Model loaded')
print(f'  Embedding dimension: {embed_model.get_sentence_embedding_dimension()}')

Loading all-MiniLM-L6-v2... (downloads ~90MB the first time only)


c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lojyn\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7878.57it/s]


Model loaded
  Embedding dimension: 384


C:\Users\lojyn\AppData\Local\Temp\ipykernel_37752\329308021.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'  Embedding dimension: {embed_model.get_sentence_embedding_dimension()}')


In [6]:
def build_embedding_text(df: pd.DataFrame) -> list[str]:
    """
    Combine instruction + response into one string per row for embedding.
    Joint encoding captures both the question intent AND the answer context,
    which improves retrieval quality for RAG.

    Args:
        df: DataFrame with 'instruction_clean' and 'response_clean' columns

    Returns:
        List of combined text strings, one per row
    """
    return (
        df['instruction_clean'].fillna('') + ' [SEP] ' + df['response_clean'].fillna('')
    ).tolist()


def generate_embeddings(texts: list[str], model: SentenceTransformer,
                         batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    """
    Generate embeddings for a list of texts using a local sentence-transformers model.

    Args:
        texts      : List of strings to embed
        model      : Loaded SentenceTransformer model
        batch_size : Rows per encode() call

    Returns:
        np.ndarray of shape (len(texts), embedding_dim), L2-normalized
        (normalization lets us use inner-product search as cosine similarity)
    """
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,   # critical: enables cosine-sim via dot product
    )
    return embeddings.astype('float32')


print('Building combined instruction+response texts...')
train_texts = build_embedding_text(train_df)

print(f'Embedding {len(train_texts):,} training rows locally (CPU)...')
print('This takes roughly 2-5 minutes for ~20K rows on a typical laptop CPU.\n')

train_embeddings = generate_embeddings(train_texts, embed_model)

print(f'\nGenerated embeddings: shape = {train_embeddings.shape}')

Building combined instruction+response texts...
Embedding 17,268 training rows locally (CPU)...
This takes roughly 2-5 minutes for ~20K rows on a typical laptop CPU.



Batches: 100%|██████████| 270/270 [05:47<00:00,  1.29s/it]


Generated embeddings: shape = (17268, 384)


## Step 4 — Build the FAISS index

In [7]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """
    Build a FAISS index for fast nearest-neighbour search.

    Uses IndexFlatIP (inner product) on L2-normalized vectors, which is
    mathematically equivalent to cosine similarity search. This is exact
    (not approximate) search — perfectly fine up to a few hundred thousand
    vectors, which covers this dataset comfortably.

    Args:
        embeddings: np.ndarray of shape (n_rows, dim), float32, normalized

    Returns:
        Trained FAISS index, ready for .search()
    """
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index


print('Building FAISS index...')
faiss_index = build_faiss_index(train_embeddings)
print(f'Index built: {faiss_index.ntotal:,} vectors, dimension {faiss_index.d}')

# Keep a row-lookup table aligned with the FAISS index order
train_df_reset = train_df.reset_index(drop=True)

Building FAISS index...
Index built: 17,268 vectors, dimension 384


## Step 5 — Vector search test

In [8]:
def vector_search(query: str, top_k: int = 5) -> list[dict]:
    """
    Run a pure vector (semantic) search against the FAISS index.

    Args:
        query : Natural language question
        top_k : Number of results to return

    Returns:
        List of result dicts with score, category, intent, instruction, response
    """
    query_vec = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores, indices = faiss_index.search(query_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = train_df_reset.iloc[idx]
        results.append({
            'score'      : float(score),
            'category'   : row['category'],
            'intent'     : row['intent'],
            'instruction': row['instruction_clean'],
            'response'   : row['response_clean'],
        })
    return results


test_queries = [
    'I want to cancel my order',
    'Where is my package? It has not arrived',
    'I forgot my password and cannot log in',
    'I was charged twice for the same purchase',
    'How do I return a defective product',
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f'QUERY: "{query}"')
    print('='*60)
    for i, r in enumerate(vector_search(query, top_k=3), 1):
        print(f'  #{i}  [{r["category"]} -> {r["intent"]}]  score={r["score"]:.4f}')
        print(f'       Matched : {r["instruction"]}')
        print(f'       Response: {r["response"][:120]}...')


QUERY: "I want to cancel my order"
  #1  [ORDER -> cancel_order]  score=0.8413
       Matched : question about canceling order your order number
       Response: I've understood you have a question about canceling order your order number. How can I assist you?...
  #2  [ORDER -> cancel_order]  score=0.8227
       Matched : i do not know how to cancel order your order number
       Response: I get it that you're unsure about how to cancel your order with order number your order number. No worries, I'm here to ...
  #3  [ORDER -> cancel_order]  score=0.8213
       Matched : cancel order your order number
       Response: I've realized that you want to cancel your order with the order number your order number. Our top priority is to provide...

QUERY: "Where is my package? It has not arrived"
  #1  [DELIVERY -> delivery_period]  score=0.5861
       Matched : where can I see when my package is going to arrive?
       Response: We understand your eagerness to track the progress and estimat

## Step 6 — Hybrid search (vector + BM25 keyword)

In [9]:
from rank_bm25 import BM25Okapi
import re

def simple_tokenize(text: str) -> list[str]:
    """Lightweight tokenizer for BM25 — lowercase, strip punctuation."""
    return re.findall(r'\b\w+\b', text.lower())


print('Building BM25 keyword index...')
tokenized_corpus = [simple_tokenize(t) for t in train_texts]
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 index built')


def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.5) -> list[dict]:
    """
    Hybrid search combining vector similarity (FAISS) and keyword relevance (BM25)
    via weighted score fusion.

    Args:
        query : Natural language question
        top_k : Number of results to return
        alpha : Weight for vector score (1-alpha is weight for BM25 score).
                alpha=1.0 -> pure vector, alpha=0.0 -> pure keyword

    Returns:
        List of result dicts, ranked by fused score
    """
    # Vector scores over the FULL corpus (so we can fuse fairly)
    query_vec = embed_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    vec_scores, vec_indices = faiss_index.search(query_vec, len(train_texts))
    vec_score_map = dict(zip(vec_indices[0], vec_scores[0]))

    # BM25 scores over the full corpus
    bm25_scores = bm25.get_scores(simple_tokenize(query))
    # Normalize BM25 scores to [0, 1] for fair fusion with cosine sim
    bm25_max = bm25_scores.max() if bm25_scores.max() > 0 else 1.0
    bm25_norm = bm25_scores / bm25_max

    fused_scores = []
    for idx in range(len(train_texts)):
        v = vec_score_map.get(idx, 0.0)
        b = bm25_norm[idx]
        fused_scores.append(alpha * v + (1 - alpha) * b)

    top_indices = np.argsort(fused_scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        row = train_df_reset.iloc[idx]
        results.append({
            'score'      : float(fused_scores[idx]),
            'category'   : row['category'],
            'intent'     : row['intent'],
            'instruction': row['instruction_clean'],
            'response'   : row['response_clean'],
        })
    return results


# --- Compare vector vs hybrid on the same query ---
comparison_query = 'I need to change the address on my order before it ships'

print(f'Query: "{comparison_query}"\n')

print('-- Vector search results --')
for r in vector_search(comparison_query, top_k=3):
    print(f'  [{r["intent"]}]  score={r["score"]:.4f}  ->  {r["instruction"]}')

print('\n-- Hybrid search results --')
for r in hybrid_search(comparison_query, top_k=3):
    print(f'  [{r["intent"]}]  score={r["score"]:.4f}  ->  {r["instruction"]}')

Building BM25 keyword index...
BM25 index built
Query: "I need to change the address on my order before it ships"

-- Vector search results --
  [change_shipping_address]  score=0.7836  ->  where to change the address
  [change_shipping_address]  score=0.7779  ->  I have got to modify my damn address, how to do it?
  [change_shipping_address]  score=0.7681  ->  wanna change my fucking address

-- Hybrid search results --
  [change_shipping_address]  score=0.8430  ->  i try to change my shipping address
  [change_shipping_address]  score=0.8111  ->  I can't change my address
  [change_shipping_address]  score=0.8032  ->  what do I have to do to change my shipping address?


## Step 7 — Evaluation: BLEU, ROUGE, and retrieval relevance@k

We evaluate retrieval quality on the **test set** — for each test query,
we retrieve the top-k most similar training examples and check:
1. **Intent match@k** — does the retrieved intent match the query's true intent?
2. **BLEU / ROUGE** — how textually similar is the retrieved response to the
   test set's gold response (a proxy for how useful that retrieved answer
   would be if served directly)

In [10]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothing = SmoothingFunction().method1


def evaluate_retrieval(test_df: pd.DataFrame, top_k: int = 1, sample_size: int = 300) -> dict:
    """
    Evaluate retrieval quality on a sample of the test set.

    For each test query:
      1. Retrieve top_k nearest training examples via vector_search()
      2. Check if retrieved intent == true intent (intent_match@k)
      3. Compute BLEU + ROUGE between the top-1 retrieved response
         and the test set's gold response

    Args:
        test_df     : Test DataFrame (from Milestone 1)
        top_k       : How many results to retrieve per query
        sample_size : Number of test rows to evaluate (full test set is slow on CPU)

    Returns:
        dict with intent_match_rate, avg_bleu, avg_rouge1, avg_rouge2, avg_rougeL
    """
    sample = test_df.sample(min(sample_size, len(test_df)), random_state=42).reset_index(drop=True)

    intent_matches = 0
    bleu_scores, r1_scores, r2_scores, rl_scores = [], [], [], []

    for _, row in tqdm(sample.iterrows(), total=len(sample), desc=f'Evaluating top-{top_k}'):
        query = row['instruction_clean']
        gold_response = str(row['response_clean'])
        true_intent = row['intent']

        results = vector_search(query, top_k=top_k)
        retrieved_intents = [r['intent'] for r in results]

        if true_intent in retrieved_intents:
            intent_matches += 1

        top1_response = results[0]['response']

        # BLEU (word-level n-gram overlap)
        ref_tokens = nltk.word_tokenize(gold_response.lower())
        hyp_tokens = nltk.word_tokenize(top1_response.lower())
        bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing)
        bleu_scores.append(bleu)

        # ROUGE (overlap-based, more forgiving than BLEU for paraphrases)
        rouge_result = rouge.score(gold_response, top1_response)
        r1_scores.append(rouge_result['rouge1'].fmeasure)
        r2_scores.append(rouge_result['rouge2'].fmeasure)
        rl_scores.append(rouge_result['rougeL'].fmeasure)

    return {
        'sample_size'        : len(sample),
        'top_k'              : top_k,
        'intent_match_rate'  : intent_matches / len(sample),
        'avg_bleu'           : float(np.mean(bleu_scores)),
        'avg_rouge1'         : float(np.mean(r1_scores)),
        'avg_rouge2'         : float(np.mean(r2_scores)),
        'avg_rougeL'         : float(np.mean(rl_scores)),
    }


print('Running evaluation on test set (sample of 300 rows, top-1 retrieval)...\n')
eval_results = evaluate_retrieval(test_df, top_k=1, sample_size=300)

print('\n' + '='*50)
print('EVALUATION RESULTS')
print('='*50)
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f'  {k:<22} {v:.4f}')
    else:
        print(f'  {k:<22} {v}')

Running evaluation on test set (sample of 300 rows, top-1 retrieval)...



Evaluating top-1: 100%|██████████| 300/300 [00:07<00:00, 39.38it/s]


EVALUATION RESULTS
  sample_size            300
  top_k                  1
  intent_match_rate      0.9767
  avg_bleu               0.1762
  avg_rouge1             0.5217
  avg_rouge2             0.2387
  avg_rougeL             0.3502


In [11]:
# Also check intent_match@3 and @5 - useful for justifying how many
# chunks to feed the LLM in the eventual generation step (Milestone 3)
print('Comparing intent match rate at different k values...\n')

for k in [1, 3, 5]:
    result = evaluate_retrieval(test_df, top_k=k, sample_size=200)
    print(f'  top-{k}:  intent_match_rate = {result["intent_match_rate"]:.4f}')

Comparing intent match rate at different k values...



Evaluating top-1: 100%|██████████| 200/200 [00:05<00:00, 38.49it/s]


  top-1:  intent_match_rate = 0.9700


Evaluating top-3: 100%|██████████| 200/200 [00:04<00:00, 40.44it/s]


  top-3:  intent_match_rate = 0.9750


Evaluating top-5: 100%|██████████| 200/200 [00:06<00:00, 32.16it/s]

  top-5:  intent_match_rate = 0.9900


## Step 8 — Save the index, embeddings, and eval report to disk

In [12]:
# Save FAISS index
faiss.write_index(faiss_index, os.path.join(INDEX_DIR, "train_index.faiss"))

# Save the row lookup table (so we can map FAISS result indices back to text)
train_df_reset.to_csv(os.path.join(INDEX_DIR, "train_lookup.csv"), index=False)

# Save raw embeddings (useful for re-building index or analysis later)
np.save(os.path.join(INDEX_DIR, "train_embeddings.npy"), train_embeddings)

# Save BM25 corpus (tokenized) for hybrid search reload
with open(os.path.join(INDEX_DIR, "bm25_corpus.pkl"), "wb") as f:
    pickle.dump(tokenized_corpus, f)

# Save evaluation report
eval_report_path = os.path.join(INDEX_DIR, "evaluation_report.json")
with open(eval_report_path, "w") as f:
    json.dump(eval_results, f, indent=2)

print(f'Saved to {os.path.abspath(INDEX_DIR)}:')
for fname in ["train_index.faiss", "train_lookup.csv", "train_embeddings.npy",
              "bm25_corpus.pkl", "evaluation_report.json"]:
    path = os.path.join(INDEX_DIR, fname)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {fname:<26} ({size_kb:,.0f} KB)')

Saved to c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\data\faiss_index:
  train_index.faiss          (25,902 KB)
  train_lookup.csv           (23,556 KB)
  train_embeddings.npy       (25,902 KB)
  bm25_corpus.pkl            (15,029 KB)
  evaluation_report.json     (0 KB)


## Milestone 2 (local) — Summary

| Deliverable | Status | File |
|---|---|---|
| Trained RAG retrieval pipeline | Done | `data/faiss_index/train_index.faiss` |
| Embeddings | Done | `data/faiss_index/train_embeddings.npy` |
| Model evaluation report | Done | `data/faiss_index/evaluation_report.json` |
| Hybrid search (vector + BM25) | Done | this notebook, Step 6 |

**Why FAISS + sentence-transformers instead of Azure:**
- Zero cost, zero account setup, fully offline
- Same retrieval interface (`vector_search`, `hybrid_search`) — easy to
  swap back to Azure later by changing only the backend, not the calling code

**Migrating to Azure later:** once your Azure issue is resolved, you can
reuse `train_df`, `train_embeddings`, and the same retrieval logic — just
point `vector_search()` at an Azure `SearchClient` instead of the FAISS index.
Your embedding *dimension* will change (384 -> 3072 if you switch to OpenAI's
`text-embedding-3-large`), so the index would need rebuilding, but the
pipeline shape stays identical.

---
**Next -> Milestone 3:** Build the RAG generation chain - retrieve top-k
context, feed it + the query into an LLM (local via Hugging Face, or API-based),
and generate a final answer.